# 02 — Exploratory Data Analysis


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Distribution Plots

Histograms of compile time, SLOC, header depth, preprocessed size, object size, git changes. Overlay codegen vs authored.


In [ ]:
# Split file-level data by origin
authored = file_df[~file_df['is_generated']]
codegen = file_df[file_df['is_generated']]

metrics = [
    ('compile_time_ms', 'Compile Time (ms)', True),
    ('code_lines', 'Source Lines of Code', True),
    ('header_max_depth', 'Max Header Depth', False),
    ('preprocessed_bytes', 'Preprocessed Size (bytes)', True),
    ('object_size_bytes', 'Object Size (bytes)', True),
    ('git_commit_count', 'Git Commit Count', True),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, label, use_log) in zip(axes, metrics):
    kw = dict(stat='density', alpha=0.6, bins=40)
    if use_log:
        kw['log_scale'] = True

    auth_data = authored[col].replace(0, pd.NA).dropna() if use_log else authored[col].dropna()
    code_data = codegen[col].replace(0, pd.NA).dropna() if use_log else codegen[col].dropna()

    if not auth_data.empty:
        sns.histplot(auth_data, ax=ax, color='steelblue', label='Authored', **kw)
    if not code_data.empty:
        sns.histplot(code_data, ax=ax, color='darkorange', label='Codegen', **kw)

    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('File-Level Metric Distributions: Authored vs Codegen', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Total files: {len(file_df)} (authored: {len(authored)}, codegen: {len(codegen)})")

## Pareto Analysis

Which 20% of targets account for 80% of compile time? Codegen Pareto.


In [ ]:
def pareto_plot(ax, df, value_col, title, label_targets=False):
    """Draw a Pareto bar chart with cumulative % line on twin axis."""
    sorted_df = df.sort_values(value_col, ascending=False).reset_index(drop=True)
    total = sorted_df[value_col].sum()
    if total == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
        ax.set_title(title)
        return 0, 0

    sorted_df['cumulative_pct'] = sorted_df[value_col].cumsum() / total * 100

    ax.bar(range(len(sorted_df)), sorted_df[value_col], color='steelblue', width=1.0)
    ax.set_ylabel('Time (ms)')
    ax.set_xlabel('Target rank')

    ax2 = ax.twinx()
    ax2.plot(range(len(sorted_df)), sorted_df['cumulative_pct'], color='darkorange', linewidth=2)
    ax2.axhline(80, color='red', linestyle='--', alpha=0.7, label='80%')
    ax2.set_ylabel('Cumulative %')
    ax2.set_ylim(0, 105)
    ax2.legend(loc='center right')

    # Find 80% cutoff
    cutoff_idx = (sorted_df['cumulative_pct'] >= 80).idxmax()
    ax.axvline(cutoff_idx, color='red', linestyle=':', alpha=0.7)

    if label_targets and len(sorted_df) <= 20:
        ax.set_xticks(range(len(sorted_df)))
        ax.set_xticklabels(sorted_df['cmake_target'], rotation=45, ha='right', fontsize=8)

    ax.set_title(title)
    return cutoff_idx + 1, len(sorted_df)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# All targets by compile time
n_80, n_total = pareto_plot(
    ax1, target_df, 'compile_time_sum_ms',
    'Pareto: Compile Time by Target',
)

# Codegen targets by codegen compile time
codegen_targets = target_df[target_df['codegen_file_count'] > 0].copy()
pareto_plot(
    ax2, codegen_targets, 'codegen_compile_time_sum_ms',
    'Pareto: Codegen Compile Time by Codegen Target',
    label_targets=True,
)

plt.tight_layout()
plt.show()

if n_total > 0:
    print(f"Top {n_80} targets ({n_80 / n_total:.1%}) account for 80% of total compile time.")
print(f"Targets with codegen: {len(codegen_targets)}")

## Correlation Analysis

Scatter plots: SLOC vs compile time, header depth vs preprocessed size, expansion ratio vs compile time.


In [ ]:
palette = {False: 'steelblue', True: 'darkorange'}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. SLOC vs Compile Time (file-level)
ax = axes[0, 0]
plot_df = file_df[['code_lines', 'compile_time_ms', 'is_generated']].dropna()
plot_df = plot_df[(plot_df['code_lines'] > 0) & (plot_df['compile_time_ms'] > 0)]
sns.scatterplot(data=plot_df, x='code_lines', y='compile_time_ms', hue='is_generated',
                palette=palette, alpha=0.5, ax=ax, legend=True)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Source Lines of Code'); ax.set_ylabel('Compile Time (ms)')
ax.set_title('SLOC vs Compile Time')

# 2. Header Depth vs Preprocessed Size (file-level)
ax = axes[0, 1]
plot_df = file_df[['header_max_depth', 'preprocessed_bytes', 'is_generated']].dropna()
plot_df = plot_df[plot_df['preprocessed_bytes'] > 0]
sns.scatterplot(data=plot_df, x='header_max_depth', y='preprocessed_bytes', hue='is_generated',
                palette=palette, alpha=0.5, ax=ax)
ax.set_yscale('log')
ax.set_xlabel('Max Header Depth'); ax.set_ylabel('Preprocessed Size (bytes)')
ax.set_title('Header Depth vs Preprocessed Size')

# 3. Git Churn vs Dependant Count (target-level)
ax = axes[0, 2]
plot_df = target_df[['git_churn_total', 'transitive_dependant_count', 'codegen_file_count']].copy()
plot_df = plot_df[(plot_df['git_churn_total'] > 0) & (plot_df['transitive_dependant_count'] > 0)]
plot_df['has_codegen'] = plot_df['codegen_file_count'] > 0
sns.scatterplot(data=plot_df, x='git_churn_total', y='transitive_dependant_count', hue='has_codegen',
                palette=palette, alpha=0.6, ax=ax)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Git Churn (lines)'); ax.set_ylabel('Transitive Dependant Count')
ax.set_title('Git Churn vs Dependant Count')

# 4. Preprocessed Size vs Template Instantiation Time (file-level)
ax = axes[1, 0]
plot_df = file_df[['preprocessed_bytes', 'gcc_template_instantiation_ms', 'is_generated']].dropna()
plot_df = plot_df[(plot_df['preprocessed_bytes'] > 0) & (plot_df['gcc_template_instantiation_ms'] > 0)]
sns.scatterplot(data=plot_df, x='preprocessed_bytes', y='gcc_template_instantiation_ms', hue='is_generated',
                palette=palette, alpha=0.5, ax=ax)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Preprocessed Size (bytes)'); ax.set_ylabel('Template Instantiation Time (ms)')
ax.set_title('Preprocessed Size vs Template Instantiation')

# 5. Expansion Ratio vs Compile Time (file-level)
ax = axes[1, 1]
plot_df = file_df[['expansion_ratio', 'compile_time_ms', 'is_generated']].dropna()
plot_df = plot_df[(plot_df['expansion_ratio'] > 0) & (plot_df['compile_time_ms'] > 0)]
sns.scatterplot(data=plot_df, x='expansion_ratio', y='compile_time_ms', hue='is_generated',
                palette=palette, alpha=0.5, ax=ax)
ax.set_yscale('log')
ax.set_xlabel('Expansion Ratio (preprocessed / source)'); ax.set_ylabel('Compile Time (ms)')
ax.set_title('Expansion Ratio vs Compile Time')

# 6. Codegen Output Volume vs Downstream Compile Time (target-level)
ax = axes[1, 2]
codegen_tgts = target_df[target_df['codegen_file_count'] > 0].copy()
if not codegen_tgts.empty:
    # Compute downstream compile cost: sum compile_time_sum_ms of all transitive dependants
    target_compile = target_df.set_index('cmake_target')['compile_time_sum_ms']
    downstream_cost = []
    for t in codegen_tgts['cmake_target']:
        dependants = nx.ancestors(G, t) if t in G else set()  # predecessors = things that depend on t
        downstream_cost.append(sum(target_compile.get(d, 0) for d in dependants))
    codegen_tgts = codegen_tgts.copy()
    codegen_tgts['downstream_compile_ms'] = downstream_cost
    plot_df = codegen_tgts[codegen_tgts['code_lines_generated'] > 0]
    sns.scatterplot(data=plot_df, x='code_lines_generated', y='downstream_compile_ms',
                    alpha=0.6, color='darkorange', ax=ax)
    ax.set_xlabel('Generated SLOC'); ax.set_ylabel('Downstream Compile Time (ms)')
else:
    ax.text(0.5, 0.5, 'No codegen targets', transform=ax.transAxes, ha='center')
ax.set_title('Codegen Output SLOC vs Downstream Compile Time')

fig.suptitle('Correlation Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## GCC Phase Breakdown

Stacked bar charts showing parsing, template instantiation, codegen, optimisation per target.


In [ ]:
gcc_cols = ['gcc_parse_time_sum_ms', 'gcc_template_time_sum_ms',
            'gcc_codegen_phase_sum_ms', 'gcc_optimization_time_sum_ms']
phase_labels = {'gcc_parse_time_sum_ms': 'Parse',
                'gcc_template_time_sum_ms': 'Template Inst.',
                'gcc_codegen_phase_sum_ms': 'Code Gen',
                'gcc_optimization_time_sum_ms': 'Optimisation'}
phase_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# Filter to targets with GCC phase data, take top 30 by compile time
gcc_df = target_df[target_df[gcc_cols].sum(axis=1) > 0].copy()
gcc_df = gcc_df.sort_values('compile_time_sum_ms', ascending=False).head(30)
gcc_df = gcc_df.set_index('cmake_target')

if gcc_df.empty:
    print("No GCC phase timing data available.")
else:
    fig, ax = plt.subplots(figsize=(14, max(6, len(gcc_df) * 0.4)))
    gcc_df[gcc_cols].rename(columns=phase_labels).plot(
        kind='barh', stacked=True, ax=ax, color=phase_colors,
    )
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('')
    ax.set_title('GCC Phase Breakdown per Target (top 30 by compile time)')
    ax.legend(loc='lower right')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Identify template-heavy targets
    gcc_total = gcc_df[gcc_cols].sum(axis=1)
    gcc_df['template_ratio'] = gcc_df['gcc_template_time_sum_ms'] / gcc_total
    template_heavy = gcc_df[gcc_df['template_ratio'] > 0.4].index.tolist()

    if template_heavy:
        print(f"\nTemplate-heavy targets (>40% template instantiation time):")
        for t in template_heavy:
            print(f"  {t}: {gcc_df.loc[t, 'template_ratio']:.1%}")
    else:
        print("\nNo targets with >40% template instantiation time.")

## Dependency Structure Overview

Degree distribution, DAG depth, maximum parallelism. Highlight codegen targets.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Degree Distribution
ax = axes[0, 0]
in_deg = pd.Series(dict(G.in_degree()))
out_deg = pd.Series(dict(G.out_degree()))
sns.histplot(in_deg, bins=30, ax=ax, color='steelblue', label='In-degree (dependants)', alpha=0.6)
sns.histplot(out_deg, bins=30, ax=ax, color='darkorange', label='Out-degree (dependencies)', alpha=0.6)
ax.set_xlabel('Degree'); ax.set_ylabel('Count')
ax.set_title('Degree Distribution')
ax.legend()

# 2. DAG Depth Distribution
ax = axes[0, 1]
depths = target_df['topological_depth'].dropna()
sns.histplot(depths, bins=max(1, int(depths.max()) + 1) if not depths.empty else 10,
             ax=ax, color='#55A868')
if not depths.empty:
    max_depth = int(depths.max())
    ax.axvline(max_depth, color='red', linestyle='--', label=f'Max depth: {max_depth}')
    ax.legend()
ax.set_xlabel('Topological Depth'); ax.set_ylabel('Count')
ax.set_title('DAG Depth Distribution')

# 3. Targets per DAG Level (parallelism potential)
ax = axes[1, 0]
depth_counts = target_df.groupby('topological_depth').size().reset_index(name='target_count')
codegen_counts = (target_df[target_df['codegen_file_count'] > 0]
                  .groupby('topological_depth').size().reset_index(name='codegen_count'))
level_df = depth_counts.merge(codegen_counts, on='topological_depth', how='left').fillna(0)

ax.bar(level_df['topological_depth'], level_df['target_count'],
       label='All targets', color='steelblue', alpha=0.7)
ax.bar(level_df['topological_depth'], level_df['codegen_count'],
       label='Codegen targets', color='darkorange', alpha=0.9)

widest_idx = level_df['target_count'].idxmax()
widest_level = int(level_df.loc[widest_idx, 'topological_depth'])
max_parallel = int(level_df.loc[widest_idx, 'target_count'])
ax.annotate(f'Max parallelism: {max_parallel}\n(level {widest_level})',
            xy=(widest_level, max_parallel), xytext=(widest_level + 1, max_parallel * 0.9),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
ax.set_xlabel('DAG Level'); ax.set_ylabel('Target Count')
ax.set_title('Targets per DAG Level (potential parallelism)')
ax.legend()

# 4. Fan-in vs Fan-out Scatter
ax = axes[1, 1]
scatter_df = target_df[['fan_out', 'fan_in', 'betweenness_centrality', 'codegen_file_count']].copy()
scatter_df = scatter_df[(scatter_df['fan_out'] > 0) | (scatter_df['fan_in'] > 0)]
scatter_df['has_codegen'] = scatter_df['codegen_file_count'] > 0
if not scatter_df.empty:
    sns.scatterplot(data=scatter_df, x='fan_out', y='fan_in', hue='has_codegen',
                    palette={False: 'steelblue', True: 'darkorange'},
                    size='betweenness_centrality', sizes=(20, 300), alpha=0.7, ax=ax)
    if scatter_df['fan_out'].max() > 10:
        ax.set_xscale('symlog', linthresh=1)
    if scatter_df['fan_in'].max() > 10:
        ax.set_yscale('symlog', linthresh=1)
ax.set_xlabel('Fan-out (dependencies)'); ax.set_ylabel('Fan-in (dependants)')
ax.set_title('Fan-in vs Fan-out (size = betweenness centrality)')

fig.suptitle('Dependency Structure Overview', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")
print(f"Max DAG depth: {int(depths.max()) if not depths.empty else 'N/A'}")
print(f"Max parallel targets at one level: {max_parallel} (level {widest_level})")
print(f"Codegen targets: {int((target_df['codegen_file_count'] > 0).sum())}")

## Codegen Overview

Summary: total codegen files, SLOC, compile time, % of total build cost.


In [ ]:
# Summary statistics
total_files = len(file_df)
codegen_files = int(file_df['is_generated'].sum())
total_sloc = int(file_df['code_lines'].sum())
codegen_sloc = int(file_df.loc[file_df['is_generated'], 'code_lines'].sum())
total_compile_ms = int(file_df['compile_time_ms'].sum())
codegen_compile_ms = int(file_df.loc[file_df['is_generated'], 'compile_time_ms'].sum())
codegen_step_ms = int(target_df['codegen_time_ms'].sum())
total_build_ms = int(target_df['total_build_time_ms'].sum())

def pct(part, whole):
    return f"{part / whole:.1%}" if whole > 0 else "N/A"

print("=" * 65)
print(f"{'Metric':<30} {'Total':>10} {'Codegen':>10} {'%':>8}")
print("-" * 65)
print(f"{'Source files':<30} {total_files:>10,} {codegen_files:>10,} {pct(codegen_files, total_files):>8}")
print(f"{'SLOC':<30} {total_sloc:>10,} {codegen_sloc:>10,} {pct(codegen_sloc, total_sloc):>8}")
print(f"{'Compile time (ms)':<30} {total_compile_ms:>10,} {codegen_compile_ms:>10,} {pct(codegen_compile_ms, total_compile_ms):>8}")
print(f"{'Codegen step time (ms)':<30} {'—':>10} {codegen_step_ms:>10,} {pct(codegen_step_ms, total_build_ms):>8}")
print("=" * 65)

# Charts
codegen_targets_df = target_df[target_df['codegen_file_count'] > 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Donut: Files
axes[0].pie(
    [codegen_files, total_files - codegen_files],
    labels=['Codegen', 'Authored'], colors=['darkorange', 'steelblue'],
    autopct='%1.1f%%', startangle=90, wedgeprops={'width': 0.5},
)
axes[0].set_title('Source Files')

# Donut: Compile Time
axes[1].pie(
    [codegen_compile_ms, total_compile_ms - codegen_compile_ms],
    labels=['Codegen', 'Authored'], colors=['darkorange', 'steelblue'],
    autopct='%1.1f%%', startangle=90, wedgeprops={'width': 0.5},
)
axes[1].set_title('Compile Time')

# Bar: Top targets by codegen ratio
if not codegen_targets_df.empty:
    top_codegen = codegen_targets_df.sort_values('codegen_ratio', ascending=False).head(20)
    sns.barplot(data=top_codegen, y='cmake_target', x='codegen_ratio',
                ax=axes[2], color='darkorange', orient='h')
    axes[2].axvline(codegen_targets_df['codegen_ratio'].mean(), color='red',
                    linestyle='--', label='Mean')
    axes[2].set_xlim(0, 1)
    axes[2].set_xlabel('Codegen Ratio (generated / total files)')
    axes[2].set_title('Top Targets by Codegen Ratio')
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, 'No codegen targets', transform=axes[2].transAxes, ha='center')
    axes[2].set_title('Top Targets by Codegen Ratio')

fig.suptitle('Codegen Build Footprint Overview', fontsize=14)
plt.tight_layout()
plt.show()